In [ ]:
!pip install -q plotly ipywidgets


In [ ]:
import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output


In [ ]:
packer_net = 65_000
manager_net = 60_000
courier_net = 57_500
service_price = 1500

ndfl_rate = 0.13
insurance_rate = 0.302

total_net_salary = packer_net + manager_net + courier_net
total_gross_salary = total_net_salary / (1 - ndfl_rate)
total_staff_cost = total_gross_salary * (1 + insurance_rate)

box_cost = 169
paper_roll_price = 179
paper_uses = 2
ribbon_roll_price = 109
ribbon_uses = 18
materials_cost_per_order = box_cost + paper_roll_price / paper_uses + ribbon_roll_price / ribbon_uses

services_df = pd.DataFrame([
    ["opakovali.online", "20x20 подарок", 1020],
    ["opakovali.online", "30x30 подарок", 1710],
    ["opakovali.online", "40x40 подарок", 2510],
    ["aflowermoskva", "упаковка min", 950],
    ["aflowermoskva", "упаковка max", 1300],
    ["moi-tvoi", "дизайн 1", 650],
    ["moi-tvoi", "дизайн 2", 570],
    ["moi-tvoi", "дизайн 3", 620],
    ["moi-tvoi", "дизайн 4", 700]
], columns=["source", "service", "price"])


In [ ]:
def calculate_unit_economics(orders_per_month):
    staff_cost_per_order = total_staff_cost / orders_per_month
    total_cost_per_order = staff_cost_per_order + materials_cost_per_order

    monthly_revenue = service_price * orders_per_month
    monthly_cost = total_staff_cost + materials_cost_per_order * orders_per_month
    monthly_profit = monthly_revenue - monthly_cost
    profit_per_order = service_price - total_cost_per_order

    return {
        "orders_per_month": orders_per_month,
        "staff_cost_per_order": staff_cost_per_order,
        "materials_cost_per_order": materials_cost_per_order,
        "total_cost_per_order": total_cost_per_order,
        "profit_per_order": profit_per_order,
        "monthly_revenue": monthly_revenue,
        "monthly_cost": monthly_cost,
        "monthly_profit": monthly_profit
    }


In [ ]:
orders_slider = widgets.IntSlider(value=300, min=50, max=1000, step=50, description="Заказы/мес:", continuous_update=False)
output = widgets.Output()

def update_dashboard(change=None):
    with output:
        clear_output(wait=True)
        orders = orders_slider.value
        result = calculate_unit_economics(orders)

        print(f"Количество заказов в месяц: {orders}")
        print(f"Цена услуги: {service_price:,.0f} ₽")
        print(f"Полная стоимость команды: {total_staff_cost:,.0f} ₽/мес.")
        print(f"Материалы на 1 заказ: {materials_cost_per_order:,.0f} ₽")
        print(f"Себестоимость 1 заказа: {result['total_cost_per_order']:,.0f} ₽")
        print(f"Прибыль с заказа: {result['profit_per_order']:,.0f} ₽")
        print(f"Месячная прибыль до налогообложения: {result['monthly_profit']:,.0f} ₽")

        cost_structure = pd.DataFrame([
            ["Персонал", result["staff_cost_per_order"]],
            ["Материалы", result["materials_cost_per_order"]]
        ], columns=["cost_type", "value"])
        px.pie(cost_structure, names="cost_type", values="value", title="Структура себестоимости одного заказа").show()

        orders_range = list(range(50, 1001, 50))
        scenario_df = pd.DataFrame({"orders_per_month": orders_range})
        scenario_df["revenue"] = scenario_df["orders_per_month"] * service_price
        scenario_df["cost"] = total_staff_cost + scenario_df["orders_per_month"] * materials_cost_per_order
        scenario_df["profit"] = scenario_df["revenue"] - scenario_df["cost"]

        scenario_long = scenario_df.melt(id_vars="orders_per_month", value_vars=["revenue", "cost", "profit"], var_name="metric", value_name="value")
        px.line(scenario_long, x="orders_per_month", y="value", color="metric", title="Выручка, затраты и прибыль в зависимости от количества заказов").show()

        fig_profit = px.line(scenario_df, x="orders_per_month", y="profit", title="Прибыль проекта в зависимости от количества заказов")
        fig_profit.add_hline(y=0)
        fig_profit.show()

orders_slider.observe(update_dashboard, names="value")
display(orders_slider, output)
update_dashboard()
